# LexAI retrieval experiment

Open this notebook with the project directory as the working directory. The first code cell downloads the public legal PDFs from the URLs in `download_sources.py` and caches them in `data/legal/`. One official Gazette copy of the Bharatiya Nyaya Sanhita replaces the three ambiguously named local copies.

The later cells reproduce the original clustering and LangGraph experiment and require provider keys in `.env`. To index PDFs for the browser app, use `python ingest.py data/legal --ocr` as described in the README.

In [ ]:
from download_sources import download_sources

pdf_files = download_sources()
for pdf in pdf_files:
    print(pdf.name)

In [ ]:
%pip install -r requirement.txt langchain-community langchain-text-splitters langgraph google-search-results

In [ ]:
from langchain_community.document_loaders import PyPDFLoader

raw_docs = []
for pdf in pdf_files:
    raw_docs.extend(PyPDFLoader(str(pdf)).load())
print(f"Loaded {len(raw_docs)} pages from {len(pdf_files)} PDFs.")

In [ ]:
# Extract + split
full_text = " ".join([d.page_content for d in raw_docs])
sentences = [s.strip() for s in full_text.split(".") if len(s.strip()) > 15]
print(f"Total sentences: {len(sentences)}")

In [ ]:
# Embed
from sentence_transformers import SentenceTransformer
model = SentenceTransformer('all-MiniLM-L6-v2')
embeddings = model.encode(sentences, show_progress_bar=True, batch_size=64)

In [ ]:
# Semantic chunk
from sklearn.metrics.pairwise import cosine_similarity


semantic_chunks = []
current_chunk = [sentences[0]]
for i in range(1, len(sentences)):
    sim = cosine_similarity([embeddings[i-1]], [embeddings[i]])[0][0]
    if sim > 0.75:
        current_chunk.append(sentences[i])
    else:
        semantic_chunks.append(". ".join(current_chunk) + ".")
        current_chunk = [sentences[i]]
semantic_chunks.append(". ".join(current_chunk) + ".")
print(f"\n✅ Total semantic chunks: {len(semantic_chunks)}")

In [ ]:
# STEP 5: Recursive Splitting (Size Control)
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(chunk_size=200, chunk_overlap=50)
final_chunks = []
for chunk in semantic_chunks:
    final_chunks.extend(splitter.split_text(chunk))
print(f"✅ Step 5: Final Hybrid Chunks: {len(final_chunks)}")
print(f"\n--- SAMPLE CHUNK ---\n{final_chunks[0]}")

In [ ]:
from sklearn.cluster import KMeans
import numpy as np

# 1. Embed the final_chunks (this will take a couple of minutes for 16,000 chunks)
print("Generating final embeddings for clustering...")
final_embeddings = model.encode(final_chunks, show_progress_bar=True, batch_size=64)

# 2. Set up K-Means
num_clusters = 20  # You can change this to 10 or 20 if you want finer topics
print(f"Clustering into {num_clusters} groups...")
kmeans = KMeans(n_clusters=num_clusters, random_state=42)

# 3. Fit the model and get labels
cluster_labels = kmeans.fit_predict(final_embeddings)
centroids = kmeans.cluster_centers_

print("\n✅ Clustering Complete!")

# 4. Show a sample of what ended up in each cluster
for i in range(num_clusters):
    print(f"\n--- CLUSTER {i} SAMPLE ---")
    # Find the indices of chunks in this cluster
    cluster_indices = np.where(cluster_labels == i)[0]
    
    # Print the first chunk as an example
    if len(cluster_indices) > 0:
        sample_idx = cluster_indices[0]
        print(f"({len(cluster_indices)} chunks in this cluster)")
        print(final_chunks[sample_idx][:200] + "...")


In [ ]:
import uuid

vector_store = []

for i, chunk in enumerate(final_chunks):
    vector_store.append({
        "id": str(uuid.uuid4()),
        "text": chunk,
        # changed 'embeddings' to 'final_embeddings' here!
        "embedding": final_embeddings[i].tolist(), 
        "cluster_id": int(cluster_labels[i])
    })

print(f"✅ Created vector store with {len(vector_store)} items.")
print("\n--- SAMPLE ITEM ---")
print("ID:", vector_store[0]["id"])
print("Text:", vector_store[0]["text"][:50], "...")
print("Embedding length:", len(vector_store[0]["embedding"]))
print("Cluster ID:", vector_store[0]["cluster_id"])


In [ ]:
cluster_store = []

for i, centroid in enumerate(centroids):
    cluster_store.append({
        "cluster_id": i,
        "centroid": centroid.tolist()
    })

In [ ]:
print(f"✅ Saved {len(cluster_store)} cluster centroids.")

In [ ]:
import os
from dotenv import load_dotenv
from pinecone import Pinecone, ServerlessSpec

# 1. Load the .env file
load_dotenv()

# Get the API key (make sure your .env has: PINECONE_API_KEY=your_actual_key_here)
api_key = os.getenv("PINECONE_API_KEY")

if not api_key:
    raise ValueError("PINECONE_API_KEY not found in .env file!")

# 2. Initialize Pinecone securely
pc = Pinecone(api_key=api_key)

index_name = "law-suggestion-index"
dimension = 384  # 'all-MiniLM-L6-v2' outputs 384-dimensional embeddings

# 3. Create the index if it doesn't exist
if not pc.indexes.exists(index_name):
    print(f"Creating new index: {index_name}...")
    pc.indexes.create(
        name=index_name,
        dimension=dimension,
        metric="cosine", # We used cosine_similarity earlier
        spec=ServerlessSpec(
            cloud="aws",
            region="us-east-1" # Default free tier region
        )
    )

# Connect to the index
index = pc.index(index_name)
print(f"✅ Connected to index '{index_name}'")

# 4. Format the data for Pinecone
pinecone_data = []
for item in vector_store:
    pinecone_data.append(
        (
            item["id"], 
            item["embedding"], 
            {
                "text": item["text"],             
                "cluster_id": item["cluster_id"]  
            }
        )
    )

# 5. Upsert (upload) in batches of 100
batch_size = 100
print(f"Uploading {len(pinecone_data)} vectors in batches of {batch_size}...")

for i in range(0, len(pinecone_data), batch_size):
    batch = pinecone_data[i : i + batch_size]
    index.upsert(vectors=batch)
    if i % 1000 == 0 and i > 0:
        print(f"Uploaded {i} vectors...")

print("🎉 Successfully uploaded all data to Pinecone!")

In [ ]:
from langgraph.graph import StateGraph, END
from typing import TypedDict
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from serpapi import GoogleSearch


# STATE
class GraphState(TypedDict):
    query: str
    domain: str
    refined_query: str
    risk: str
    context: str
    score: float
    roadmap: str
    final_answer: str

In [ ]:
import os
from dotenv import load_dotenv
from groq import Groq

# Load environment variables
load_dotenv()

# Get API key
GROQ_API_KEY = os.getenv("GROQ_API_KEY")

# Initialize client
client = Groq(api_key=GROQ_API_KEY)

In [ ]:
# LLM FUNCTION (GROQ)

def call_llm(prompt):
    response = client.chat.completions.create(
        model=os.getenv("GROQ_MODEL", "openai/gpt-oss-120b"),
        messages=[
            {"role": "system", "content": "You are a helpful legal AI assistant."},
            {"role": "user", "content": prompt}
        ],
        temperature=0.3
    )
    return response.choices[0].message.content.strip()

In [ ]:
# 1. DOMAIN CLASSIFIER

def domain_classifier(state):
    query = state["query"].lower()

    # Expanded keyword set covering common Indian legal areas
    law_keywords = [
        # Criminal & Police
        "law", "ipc", "crpc", "bns", "bnss", "bsa", "fir", "police", "arrest", 
        "bail", "jail", "convict", "accused", "suspect", "offence", "crime",
        
        # Courts & Procedure
        "court", "judge", "magistrate", "justice", "litigation", "case", "suit",
        "petition", "writ", "affidavit", "summon", "warrant", "bench", "registry",
        
        # Professionals
        "legal", "lawyer", "advocate", "counsel", "solicitor", "attorney",
        
        # Family & Personal
        "divorce", "marriage", "custody", "alimony", "maintenance", "succession", "will",
        
        # Property & Civil
        "property", "lease", "tenant", "rent", "registry", "agreement", "contract",
        "compensation", "damages", "debt", "recovery",
        
        # Constitutional & Rights
        "constitution", "fundamental rights", "article", "pil", "rights",
        
        # Crimes & Specific Acts
        "fraud", "scam", "harassment", "defamation", "posh", "dowry", "sc/st",
        "theft", "murder", "assault", "cybercrime", "cheating"
    ]

    # Check if ANY keyword is in the user query
    domain = "law" if any(k in query for k in law_keywords) else "general"
    
    print(f"📁 Domain Classified: {domain.upper()}")
    
    return {**state, "domain": domain}



def domain_router(state):
    return "law" if state["domain"] == "law" else "general"

In [ ]:
# 2. GENERAL LLM (NON-LAW)

def general_llm(state):
    answer = call_llm(state["query"])
    return {**state, "final_answer": answer}


In [ ]:
# 3. QUERY REFINEMENT



def refine_query(state):
    prompt = f"""
    You are an Indian Legal Research Expert.
    Convert the user SITUATION into a precise legal search query.

    CRITICAL RULES:
    - The query MUST be focused on the Indian Legal System.
    - Append keywords like "Indian Law", "IPC", "BNS", or "High Court of India" if applicable.
    - Do NOT include any US-specific terminology (like "Miranda" or "Amendment 5").

    USER SITUATION:
    {state['query']}
    
    OUTPUT:
    [Just the legal search query string]
    """

    refined = call_llm(prompt)
    print(f"🔍 Refined Query: {refined}")

    return {**state, "refined_query": refined}


In [ ]:
# 4. RISK CLASSIFICATION


def risk_classifier(state):
    """
    Classifies the user query into LOW, MEDIUM, or HIGH risk 
    using a predefined legal rubric for expert-level accuracy.
    """
    prompt = f"""
    You are a Legal Risk Assessment Expert. 
    Analyze the following user situation and classify the risk level into exactly one of these: LOW, MEDIUM, HIGH.

    RISK THRESHOLDS:
    - LOW: General informational queries about laws, basic rights, or minor civil matters with no immediate threat. Example: "What is the age for voting?" or "How do I register a birth?"
    - MEDIUM: Legal notices received, contract disputes, non-violent civil disagreements, or situations where a minor crime might have been committed but there is no immediate danger of arrest.
    - HIGH: Immediate threat of arrest, physical violence, serious criminal charges (e.g., assault, theft, harassment), or urgent legal deadlines that could lead to loss of liberty or safety.

    USER QUERY:
    {state['query']}

    INSTRUCTIONS:
    Provide your response in this format:
    RISK: [LEVEL]
    REASONING: [Brief explanation]
    """

    response = call_llm(prompt)
    
    # Robust parsing logic
    risk_raw = response.upper()
    if "HIGH" in risk_raw:
        risk = "high"
    elif "MEDIUM" in risk_raw:
        risk = "medium"
    else:
        risk = "low"

    print(f"⚖️ Risk Assessment: {risk.upper()}")
    # Optional: You can print the full reasoning if you want to see the LLM's logic
    # print(f"Reasoning: {response}") 
    
    return {**state, "risk": risk}


In [ ]:
# 5. RAG NODE 
import os
from pinecone import Pinecone
from sentence_transformers import SentenceTransformer

# Initialize Pinecone and point to your index
pc = Pinecone(api_key=os.environ.get("PINECONE_API_KEY"))
index = pc.index("law-suggestion-index") 

# Load the exact same embedding model you used earlier
model = SentenceTransformer('all-MiniLM-L6-v2')


def rag_node(state):
    query = state.get("refined_query", state["query"])

    # Embed the user query
    query_embedding = model.encode(query).tolist()

    
    # STAGE 1: Find the best matching cluster_id
    
    first_search = index.query(
        vector=query_embedding,
        top_k=1, # Just need the top 1 to find the best cluster
        include_metadata=True
    )

    best_cluster_id = None
    if first_search.get("matches") and len(first_search["matches"]) > 0:
        best_cluster_id = first_search["matches"][0]["metadata"].get("cluster_id")

    # ---------------------------------------------------------
    # STAGE 2: Deep dive into ONLY that specific cluster
    # ---------------------------------------------------------
    contexts = []
    best_score = 0.0

    if best_cluster_id is not None:
        print(f"🎯 Best cluster matched: Cluster {best_cluster_id}. Going deeper...")

        deep_search = index.query(
            vector=query_embedding,
            top_k=3, # Retrieve top 3 detailed chunks from this cluster
            filter={
                "cluster_id": {"$eq": best_cluster_id} # Apply the cluster filter
            },
            include_metadata=True
        )

        if deep_search.get("matches"):
            best_score = deep_search["matches"][0]["score"]
            for match in deep_search["matches"]:
                chunk_text = match["metadata"].get("text", "") 
                if chunk_text:
                    contexts.append(chunk_text)
    else:
        print("⚠️ No matches found in the vector database.")

    # Join the retrieved chunks into a single readable context block
    context = "\n\n---\n\n".join(contexts)

    # Return updated state
    return {**state, "context": context, "score": best_score}

In [ ]:
# 6. SEARCH WEB UTILITY & SERP NODE


def search_web(query):
    api_key = os.getenv("SERPAPI_API_KEY")
    if not api_key:
        return "Search skipped: SERPAPI_API_KEY not found."

    params = {
        "engine": "google",
        "q": query,
        "api_key": api_key,
        "num": 3,
        "gl": "in",             # RESTRICT result origin to INDIA
        "hl": "en",             # INTERFACE in English
        "google_domain": "google.co.in" # USE Indian Google domain
    }

    try:
        search = GoogleSearch(params)
        results = search.get_dict()
        organic_results = results.get("organic_results", [])
        
        web_context = ""
        for i, res in enumerate(organic_results):
            web_context += f"\nSource {i+1}: {res.get('title')}\nSnippet: {res.get('snippet')}\n"
            
        return web_context if web_context else "No relevant Indian web results found."
    except Exception as e:
        return f"Search error: {str(e)}"

def serp_node(state):
    print("🌐 Fetching latest Indian legal context from the web...")
    
    # Injected "in India" keywords as a hard-coded safety measure
    query = state.get("refined_query", state["query"])
    restricted_query = f"{query} under Indian Law BNS IPC"
    
    web_data = search_web(restricted_query)
    
    existing_context = state.get("context", "")
    new_context = f"{existing_context}\n\n=== INDIAN WEB CONTEXT ===\n{web_data}"
    
    return {**state, "context": new_context}



In [ ]:
# 8. ROADMAP NODE

def roadmap_node(state):
    """
    Generates a step-by-step legal roadmap for the user based on the context.
    """
    print("🗺️ Generating legal roadmap...")
    
    context = state.get("context", "No context available.")
    query = state["query"]
    
    prompt = f"""
    Based on the following legal context and the user's situation, generate a clear, step-by-step roadmap.
    
    User Query: {query}
    
    Legal Context:
    {context}
    
    Instructions:
    - Provide a numbered list of actions (Step 1, Step 2, etc.).
    - Keep it practical (e.g., mention filing FIRs, court procedures, or seeking specific counsel).
    - Use a professional yet supportive tone.
    - If the context is missing specific details, give general best-practice advice for this legal domain.
    """
    
    roadmap = call_llm(prompt)
    
    return {**state, "roadmap": roadmap}


In [ ]:
def rag_router(state):
    """
    Decides whether to perform a web search or go straight to the roadmap.
    """
    # Retrieve the score we updated in the rag_node
    score = state.get("score", 0.0)
    
    # If the RAG score is low, we route to SERP for web data
    if score < 0.5:
        print(f"⚠️ Low confidence ({score:.2f}). Routing to SERP...")
        return "serp"
    else:
        print(f"✅ High confidence ({score:.2f}). Moving to Roadmap...")
        return "roadmap"


In [ ]:
# 9. LEGAL EXPERT (Strictly Indian Law)
# -----------------------------

def legal_expert(state):
    """
    Synthesizes all gathered information into a FINAL expert response.
    Strictly follows Indian statutes (IPC, BNS, CrPC, etc.) and formats.
    """
    print("⚖️ Finalizing expert response (Strictly Indian Law)...")
    
    prompt = f"""
    You are a STRICT Indian legal assistant.
    Your job is to provide ONLY India-specific legal guidance.

    -------------------------------------
    CRITICAL RULES (MUST FOLLOW):

    1. ONLY use Indian laws:
       - Indian Penal Code (IPC) / Bharatiya Nyaya Sanhita (BNS)
       - Code of Criminal Procedure (CrPC) / Bharatiya Nagarik Suraksha Sanhita (BNSS)
       - Constitution of India
       - Labour laws (Code on Wages, Payment of Wages Act, etc.)
       - POSH Act, etc.

    2. NEVER include foreign laws:
       - Do NOT mention US laws (FLSA, NLRA, etc.)
       - Do NOT mention UK or other countries

    3. If unsure about the correct Indian law:
       - DO NOT guess
       - Give general guidance WITHOUT naming wrong laws

    4. Match law to query domain:
       - Salary / wages → labour laws (Code on Wages, Payment of Wages Act)
       - FIR → CrPC 154 / BNSS 173
       - Arrest → CrPC 41, 50, 57 / BNSS 35, 47, 58
       - Bail → CrPC 436-439 / BNSS 478-484
       - Workplace harassment → POSH Act

    5. DO NOT hallucinate sections or mix unrelated laws

    -------------------------------------
    CONTEXT (Retrieved Laws/Web Info):
    {state.get("context", "No direct statute context found.")}

    USER QUERY:
    {state["query"]}
    
    RISK LEVEL:
    {state["risk"].upper()}

    -------------------------------------
    TASK:
    Generate a structured response with:
    1. Disclaimer  
    2. Lawyer Requirement (Required / Optional / Not Needed)  
    3. Legal Summary (India-specific)  
    4. Relevant Laws (ONLY if correct)  
    5. Step-by-step Action Roadmap  
    6. Important Rights / Notes  
    7. Final Note  

    -------------------------------------
    IMPORTANT:
    - Ensure all laws are from INDIA only.
    - If no exact section is known → skip "Relevant Laws".
    - Keep response practical and actionable.
    - If you mention any non-Indian law, the answer is INVALID.
    """
    
    final_answer = call_llm(prompt)
    
    return {**state, "final_answer": final_answer}


In [ ]:
from langgraph.graph import END

# Map your existing function to the node name
final_llm = legal_expert 

# BUILD LANGGRAPH
# -----------------------------
graph = StateGraph(GraphState)

graph.add_node("domain", domain_classifier)
graph.add_node("general", general_llm)
graph.add_node("refine", refine_query)
graph.add_node("risk", risk_classifier)
graph.add_node("rag", rag_node)
graph.add_node("serp", serp_node)
graph.add_node("roadmap", roadmap_node)
graph.add_node("final", final_llm)

graph.set_entry_point("domain")

# Domain routing
graph.add_conditional_edges(
    "domain",
    domain_router,
    {
        "law": "refine",
        "general": "general"
    }
)

# Law pipeline
graph.add_edge("refine", "risk")
graph.add_edge("risk", "rag")

# RAG decision (Routes to SERP or Roadmap based on data quality)
graph.add_conditional_edges(
    "rag",
    rag_router,
    {
        "roadmap": "roadmap",
        "serp": "serp"
    }
)

graph.add_edge("serp", "roadmap")
graph.add_edge("roadmap", "final")

# End
graph.add_edge("final", END)
graph.add_edge("general", END)

# Compile
app = graph.compile()
print("🎉 AI Legal Agent Graph is Ready!")


In [ ]:
# RUN EXAMPLE
# -----------------------------
query = "What are the rights of women during police investigation?"

# Initializing the state with the user query
result = app.invoke({
    "query": query
})

print("\n--- AGENT EXECUTION COMPLETE ---\n")

# Display as professional Markdown
from IPython.display import Markdown
Markdown(result["final_answer"])
